# gpucc GPU Validation

**Run this on Google Colab with a T4 GPU (Runtime → Change runtime type → T4 GPU)**

This notebook:
1. Installs dependencies (CuPy)
2. Mounts / uploads the `gpucc` compiler
3. Compiles `vector_add` to PTX and shows the IR + PTX output
4. Loads the PTX via CuPy `RawModule` and runs it on the GPU
5. Validates output against NumPy reference
6. Benchmarks throughput
7. Repeats for `matmul`


In [ ]:
# ── Step 1: Install CuPy (for T4 = CUDA 12.x) ─────────────────────────────
!pip install cupy-cuda12x -q
import subprocess, sys
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# ── Step 2: Upload gpucc package ───────────────────────────────────────────
# Option A: if you have the repo as a zip, upload it:
# from google.colab import files
# uploaded = files.upload()  # upload gpucc.zip
# !unzip gpucc.zip

# Option B: clone from GitHub (if you've pushed it):
# !git clone https://github.com/YOUR_USERNAME/gpucc.git
# import sys; sys.path.insert(0, 'gpucc')

# Option C (easiest for testing): paste the package inline
# For now, assume gpucc/ is in the current directory
import sys
sys.path.insert(0, '.')  # adjust if needed

import gpucc
print('gpucc loaded successfully')

## Part 1: vector_add

Compute `c[i] = a[i] + b[i]` for 1M elements.

In [ ]:
from gpucc import kernel, float32, int32, N

@kernel
def vector_add(a: float32[N], b: float32[N], c: float32[N], n: int32):
    tid = thread_id()
    if tid < n:
        c[tid] = a[tid] + b[tid]

print('── IR ──────────────────────────────────────────────────')
print(vector_add.ir_text)
print('── PTX ─────────────────────────────────────────────────')
print(vector_add.ptx)

In [ ]:
import cupy as cp
import numpy as np
from gpucc.runtime.cupy_runner import GPURunner

# Determine actual SM version
device = cp.cuda.Device(0)
cc = device.compute_capability
sm = f'sm_{cc}'
print(f'GPU compute capability: {cc}  → compiling for {sm}')

# Recompile for actual GPU architecture
ptx = vector_add.compile(opt_level=2, sm_version=sm)

# Create runner
runner = GPURunner(ptx, 'vector_add')

# Prepare inputs
N_ELEMS = 1 << 20  # 1M elements
np_a = np.random.rand(N_ELEMS).astype(np.float32)
np_b = np.random.rand(N_ELEMS).astype(np.float32)

a_gpu = cp.array(np_a)
b_gpu = cp.array(np_b)
c_gpu = cp.zeros(N_ELEMS, dtype=np.float32)

# Launch parameters
BLOCK = 256
GRID  = (N_ELEMS + BLOCK - 1) // BLOCK

runner(a_gpu, b_gpu, c_gpu, np.int32(N_ELEMS),
       grid=(GRID,), block=(BLOCK,))

# Validate
expected = np_a + np_b
actual   = c_gpu.get()
np.testing.assert_allclose(actual, expected, rtol=1e-5)
print('✓ vector_add output matches NumPy reference')

In [ ]:
# Benchmark
import time

N_RUNS = 100
cp.cuda.Stream.null.synchronize()
t0 = time.perf_counter()
for _ in range(N_RUNS):
    runner(a_gpu, b_gpu, c_gpu, np.int32(N_ELEMS),
           grid=(GRID,), block=(BLOCK,))
cp.cuda.Stream.null.synchronize()
t1 = time.perf_counter()

elapsed_ms = (t1 - t0) / N_RUNS * 1000
bytes_moved = N_ELEMS * 3 * 4  # 3 arrays × 4 bytes
bw_gb_s = bytes_moved / (elapsed_ms / 1000) / 1e9

print(f'Average time per kernel: {elapsed_ms:.3f} ms')
print(f'Effective memory bandwidth: {bw_gb_s:.1f} GB/s')
print(f'(T4 peak: ~300 GB/s — coalesced access should get close)')

## Part 2: matmul

Naive (non-tiled) matrix multiplication. Each thread computes one element of C = A @ B.

Note: pass the inner dimension as param `k` so the compiler can find the stride for 2D indexing.

In [ ]:
from gpucc import kernel, float32, int32, N, M, K

@kernel
def matmul(A: float32[M, K], B: float32[K, N], C: float32[M, N],
           m: int32, k: int32, n: int32):
    row = block_id(0) * block_size(0) + thread_id(0)
    col = block_id(1) * block_size(1) + thread_id(1)
    acc = 0.0
    for ki in range(k):
        acc = acc + A[row, ki] * B[ki, col]
    C[row, col] = acc

print('── IR ──────────────────────────────────────────────────')
print(matmul.ir_text)

In [ ]:
ptx_matmul = matmul.compile(opt_level=2, sm_version=sm)
runner_matmul = GPURunner(ptx_matmul, 'matmul')

# Small test: 64×64 × 64×64
MM, KK, NN = 64, 64, 64
np_A = np.random.rand(MM, KK).astype(np.float32)
np_B = np.random.rand(KK, NN).astype(np.float32)

A_gpu = cp.array(np_A)
B_gpu = cp.array(np_B)
C_gpu = cp.zeros((MM, NN), dtype=np.float32)

BLOCK2 = (16, 16)
GRID2  = ((MM + BLOCK2[0] - 1) // BLOCK2[0],
          (NN + BLOCK2[1] - 1) // BLOCK2[1])

runner_matmul(
    A_gpu, B_gpu, C_gpu,
    np.int32(MM), np.int32(KK), np.int32(NN),
    grid=GRID2, block=BLOCK2,
)

expected_C = np_A @ np_B
actual_C   = C_gpu.get()
np.testing.assert_allclose(actual_C, expected_C, rtol=1e-4, atol=1e-4)
print('✓ matmul output matches NumPy @ reference')

## Part 3: Show the full compilation pipeline

This is the demo: source code → readable IR → annotated PTX → running GPU kernel.

In [ ]:
print('=' * 65)
print('STAGE 1: Source (what you write)')
print('=' * 65)
src = '''
@kernel
def vector_add(a: float32[N], b: float32[N], c: float32[N], n: int32):
    tid = thread_id()
    if tid < n:
        c[tid] = a[tid] + b[tid]
'''
print(src)

print('=' * 65)
print('STAGE 2: IR (compiler intermediate representation)')
print('=' * 65)
print(vector_add.ir_text)

print('=' * 65)
print('STAGE 3: PTX (GPU assembly your compiler generates)')
print('=' * 65)
print(vector_add.ptx)

In [ ]:
# Show optimization passes in action
# Compile a kernel with a constant expression that gets folded
from gpucc import kernel, float32, int32, N

@kernel
def scaled_add(a: float32[N], b: float32[N], c: float32[N], n: int32):
    tid = thread_id()
    # 2.0 * 3.0 should be constant-folded to 6.0
    scale = 2.0 * 3.0
    if tid < n:
        c[tid] = (a[tid] + b[tid]) * scale

print('── IR before optimization (opt_level=0) ──────────────')
ptx_raw = scaled_add.compile(opt_level=0)
print(scaled_add.ir_text)  # IR is always unoptimized (from original parse)

print('── PTX without optimization ──────────────────────────')
print(ptx_raw[:800], '...')

print('── PTX with opt_level=2 ──────────────────────────────')
ptx_opt = scaled_add.compile(opt_level=2)
print(ptx_opt[:800], '...')

# The optimized version should have 6.0 as a constant instead of mul.f32 2.0, 3.0
print()
if 'mul.f32' in ptx_raw:
    print('✓ Unoptimized: 2.0 * 3.0 emits a mul.f32 instruction')
if '6' in ptx_opt and 'mul.f32' not in ptx_opt.split('scale')[0]:
    print('✓ Optimized: constant folded to 6.0')